# Chapter 10 — Grid-Stride Loops & Element-wise Ops

> Course: **llm.c — Zero to Hero**, Chapter 10 of ~20.
> Builds on Chapter 9 (kernels, `<<<grid, block>>>`, host↔device memory).

In Chapter 9 we wrote `gelu_forward_kernel1` with **one thread per element**. That works for any `N`, as long as we launch `ceil(N/block_size)` blocks. But there's a more flexible (and often more efficient) pattern: the **grid-stride loop**. It decouples the launch geometry from the data size — you choose how many threads to launch, and *each thread handles multiple elements in a strided sweep*. It's the canonical way to write CUDA kernels for arbitrary `N`, and you'll see it in nearly every production `llm.c` kernel.

This chapter is short. We'll write one new pattern, see why production kernels use it, and run an `encoder_forward` GPU port.

### Learning objectives

By the end of this chapter you will:

- Write a **grid-stride loop**: `for (int i = tid; i < N; i += blockDim.x * gridDim.x) { ... }`.
- Explain when grid-stride beats one-thread-per-element (large `N`, fixed grid size, register reuse).
- Port `encoder_forward` (Chapter 2) to a CUDA kernel using the grid-stride pattern.


## 1. Concept — One Thread, Many Elements

In Chapter 9 we did:

```c
__global__ void kernel1(float* out, ..., int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) { ... }                    // one element per thread, then exit
}
```

A **grid-stride loop** lets each thread handle multiple elements:

```c
__global__ void kernel2(float* out, ..., int N) {
    int tid    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;          // total #threads in the grid
    for (int i = tid; i < N; i += stride) {       // <-- the grid-stride loop
        // ... do work on out[i] ...
    }
}
```

The total number of threads launched is `gridDim.x * blockDim.x`. Each thread starts at its global index `tid` and **strides forward by the grid size** until it falls off the end. So if `N = 1,000,000` and the grid has `262,144` threads, each thread handles 4 elements.

### Why prefer this?

Three real reasons:

1. **You decouple launch geometry from `N`.** The kernel works for any `N`, even `N` larger than the GPU's max grid size. Pick whatever grid is best for the GPU (typically `~ #SMs * 4` blocks) and ignore `N` entirely.
2. **Better instruction-level reuse.** A thread that runs the loop body 4 times can reuse register state, hide latency across iterations, and amortize launch overhead.
3. **Robustness.** No more "did I compute `grid_size = ceil(N/block_size)` correctly?" The loop's bound is `N`; nothing else.

`llm.c`'s production kernels use this pattern essentially everywhere.


## 2. Demo — Grid-Stride GELU on Different Grid Sizes

In [ ]:
!mkdir -p course/ch10_build


In [ ]:
%%writefile course/ch10_build/gelu_stride.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

// Grid-stride GELU: each thread processes multiple elements
__global__ void gelu_stride(float* out, const float* inp, int N) {
    int tid    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = tid; i < N; i += stride) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}

int main(void) {
    const int N = 1 << 24;       // 16M elements
    float *d_inp, *d_out;
    cudaMalloc(&d_inp, N*4); cudaMalloc(&d_out, N*4);

    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int iters = 50;

    int block_size = 256;
    int grid_sizes[] = {64, 256, 1024, 4096, 16384, 65536};
    int n_g = 6;

    printf("N=%d  block_size=%d  iters=%d\n", N, block_size, iters);
    printf("grid_size  elems/thread  ms/iter   bandwidth (GB/s)\n");
    for (int gi = 0; gi < n_g; gi++) {
        int grid_size = grid_sizes[gi];
        int total_threads = grid_size * block_size;
        // warmup
        gelu_stride<<<grid_size, block_size>>>(d_out, d_inp, N);
        cudaDeviceSynchronize();

        cudaEventRecord(s);
        for (int k = 0; k < iters; k++) gelu_stride<<<grid_size, block_size>>>(d_out, d_inp, N);
        cudaEventRecord(e); cudaEventSynchronize(e);
        float ms; cudaEventElapsedTime(&ms, s, e);
        float per = ms / iters;
        float bw = (2.0f*N*4.0f/1e9f) / (per/1000.0f);
        printf("%9d  %12d  %7.3f  %14.1f\n",
               grid_size, (N + total_threads - 1) / total_threads, per, bw);
    }
    cudaFree(d_inp); cudaFree(d_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch10_build/gelu_stride course/ch10_build/gelu_stride.cu && ./course/ch10_build/gelu_stride


You should see that **a wide range of grid sizes give similar peak bandwidth** — the kernel is correct and fast for any choice. With grid_size 64, each thread does ~1024 elements; with grid_size 65536, each thread does ~1 element. Below a certain grid size, you can't hide latency; above some threshold, more threads don't help. **Pick something in the middle (a few × the SM count) and don't sweat it.**

That's the practical payoff of grid-stride: one kernel works correctly across orders of magnitude of `N` and grid size, with near-peak performance.


## 3. Encoder Forward on the GPU

The CPU `encoder_forward` (Chapter 2) does `out[b,t,i] = wte[ix,i] + wpe[t,i]`. The whole computation has `B*T*C` independent elementwise writes — perfect for grid-stride.

Here's a kernel inspired by `dev/cuda/encoder_forward.cu` (kernel 1):

```c
__global__ void encoder_kernel(float* out, const int* inp, const float* wte, const float* wpe,
                               int B, int T, int C) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int N = B * T * C;
    if (idx < N) {
        int b  = idx / (T * C);
        int t  = (idx / C) % T;
        int c  = idx % C;
        int ix = inp[b * T + t];
        out[idx] = wte[ix*C + c] + wpe[t*C + c];
    }
}
```

The trick: instead of a `(b, t, c)` triple loop, **flatten** to a single 1-D index `idx ∈ [0, B*T*C)`, then unflatten via `/` and `%` to recover `(b, t, c)`. One thread per output element.

We can equally write it as a grid-stride loop:

```c
__global__ void encoder_kernel_stride(float* out, const int* inp, const float* wte, const float* wpe,
                                      int B, int T, int C) {
    int N = B * T * C;
    int tid    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int idx = tid; idx < N; idx += stride) {
        int b  = idx / (T * C);
        int t  = (idx / C) % T;
        int c  = idx % C;
        int ix = inp[b * T + t];
        out[idx] = wte[ix*C + c] + wpe[t*C + c];
    }
}
```

Let's run both and verify correctness against the CPU implementation.


In [ ]:
%%writefile course/ch10_build/encoder_forward.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

void encoder_forward_cpu(float* out, const int* inp, const float* wte, const float* wpe,
                         int B, int T, int C) {
    for (int b = 0; b < B; b++)
        for (int t = 0; t < T; t++) {
            float* out_bt = out + b*T*C + t*C;
            int ix = inp[b*T + t];
            for (int i = 0; i < C; i++) out_bt[i] = wte[ix*C + i] + wpe[t*C + i];
        }
}

__global__ void encoder_kernel_stride(float* out, const int* inp,
                                      const float* wte, const float* wpe,
                                      int B, int T, int C) {
    int N = B * T * C;
    int tid    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int idx = tid; idx < N; idx += stride) {
        int b  = idx / (T * C);
        int t  = (idx / C) % T;
        int c  = idx % C;
        int ix = inp[b*T + t];
        out[idx] = wte[ix*C + c] + wpe[t*C + c];
    }
}

int main(void) {
    int B = 4, T = 64, C = 128, V = 1000, maxT = 1024;
    size_t Nout = (size_t)B*T*C;

    // CPU reference data
    float* h_wte = (float*) malloc(V*C*4);
    float* h_wpe = (float*) malloc(maxT*C*4);
    int*   h_inp = (int*)   malloc(B*T*sizeof(int));
    float* h_ref = (float*) malloc(Nout*4);
    float* h_gpu = (float*) malloc(Nout*4);
    for (size_t i = 0; i < (size_t)V*C; i++) h_wte[i] = (float)((i*37) % 100) / 100.0f;
    for (size_t i = 0; i < (size_t)maxT*C; i++) h_wpe[i] = (float)((i*23) % 100) / 100.0f;
    for (int i = 0; i < B*T; i++) h_inp[i] = (i*7) % V;
    encoder_forward_cpu(h_ref, h_inp, h_wte, h_wpe, B, T, C);

    // device buffers
    float *d_wte, *d_wpe, *d_out;  int* d_inp;
    cudaMalloc(&d_wte, V*C*4); cudaMalloc(&d_wpe, maxT*C*4);
    cudaMalloc(&d_inp, B*T*sizeof(int)); cudaMalloc(&d_out, Nout*4);
    cudaMemcpy(d_wte, h_wte, V*C*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_wpe, h_wpe, maxT*C*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_inp, h_inp, B*T*sizeof(int), cudaMemcpyHostToDevice);

    encoder_kernel_stride<<<256, 256>>>(d_out, d_inp, d_wte, d_wpe, B, T, C);
    cudaMemcpy(h_gpu, d_out, Nout*4, cudaMemcpyDeviceToHost);

    float maxerr = 0;
    for (size_t i = 0; i < Nout; i++) {
        float e = fabsf(h_gpu[i] - h_ref[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("Encoder forward, B=%d T=%d C=%d  max |gpu - cpu| = %.2e\n", B, T, C, maxerr);
    cudaFree(d_wte); cudaFree(d_wpe); cudaFree(d_inp); cudaFree(d_out);
    free(h_wte); free(h_wpe); free(h_inp); free(h_ref); free(h_gpu);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch10_build/encoder_forward course/ch10_build/encoder_forward.cu && ./course/ch10_build/encoder_forward


**Bit-identical**. The GPU port of the embedding layer runs — and produces the same answer as the CPU.

Notice that the *body* of the GPU kernel is once again a near-copy of the CPU loop body, with the loop variables `(b, t, c)` derived from a flat index instead of a nested `for`. **This is the dominant porting pattern from CPU `llm.c` to CUDA `llm.c`.** Once you see it, you can read 80% of the production CUDA code by inspection.


## 4. The Translation Bridge

| Pattern | One-thread-per-element (Ch 9) | Grid-stride loop |
|---|---|---|
| Grid size | `ceil(N / block_size)` | Anything reasonable, e.g. `4 × #SMs` |
| Per-thread work | 1 element | `ceil(N / total_threads)` elements |
| Bounds check | `if (i < N)` | The `for` loop's `i < N` test |
| Best for | Small `N`, simple kernels | Large `N`, kernels with register reuse |

`llm.c` uses **both**. Tiny kernels stay one-thread-per-element; bigger ones use grid-stride. There's no universal answer.


## 5. TODO Exercise — Grid-Stride Residual

In [ ]:
%%writefile course/ch10_build/exercise1.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// TODO: write a grid-stride residual kernel.
// out[i] = inp1[i] + inp2[i]  for i in [0, N).
__global__ void residual_stride(float* out, const float* inp1, const float* inp2, int N) {
    // 1. compute tid and stride
    // 2. for (int i = tid; i < N; i += stride) out[i] = inp1[i] + inp2[i];
}

int main(void) {
    const int N = 1 << 20;
    float *h_a = (float*) malloc(N*4);
    float *h_b = (float*) malloc(N*4);
    float *h_o = (float*) malloc(N*4);
    for (int i = 0; i < N; i++) { h_a[i] = (float)i; h_b[i] = (float)(2*i); }
    float *d_a, *d_b, *d_o;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_o, N*4);
    cudaMemcpy(d_a, h_a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N*4, cudaMemcpyHostToDevice);

    // TODO: launch with grid_size 256, block_size 256
    // residual_stride<<<256, 256>>>(d_o, d_a, d_b, N);

    cudaMemcpy(h_o, d_o, N*4, cudaMemcpyDeviceToHost);
    int correct = 1;
    for (int i = 0; i < N; i++) if (h_o[i] != 3.0f*i) { correct = 0; break; }
    printf("%s\n", correct ? "PASS" : "FAIL");
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_o);
    free(h_a); free(h_b); free(h_o);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch10_build/exercise1 course/ch10_build/exercise1.cu && ./course/ch10_build/exercise1


### Solution

In [ ]:
%%writefile course/ch10_build/exercise1_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void residual_stride(float* out, const float* inp1, const float* inp2, int N) {
    int tid    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = tid; i < N; i += stride) out[i] = inp1[i] + inp2[i];
}

int main(void) {
    const int N = 1 << 20;
    float *h_a = (float*) malloc(N*4);
    float *h_b = (float*) malloc(N*4);
    float *h_o = (float*) malloc(N*4);
    for (int i = 0; i < N; i++) { h_a[i] = (float)i; h_b[i] = (float)(2*i); }
    float *d_a, *d_b, *d_o;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_o, N*4);
    cudaMemcpy(d_a, h_a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N*4, cudaMemcpyHostToDevice);
    residual_stride<<<256, 256>>>(d_o, d_a, d_b, N);
    cudaMemcpy(h_o, d_o, N*4, cudaMemcpyDeviceToHost);
    int correct = 1;
    for (int i = 0; i < N; i++) if (h_o[i] != 3.0f*i) { correct = 0; break; }
    printf("%s\n", correct ? "PASS" : "FAIL");
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_o);
    free(h_a); free(h_b); free(h_o);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch10_build/exercise1_sol course/ch10_build/exercise1_sol.cu && ./course/ch10_build/exercise1_sol


## Recap

You now know:

- The **grid-stride loop** decouples grid size from `N`. One kernel handles arbitrary input sizes correctly and near-optimally.
- The dominant CPU→GPU port pattern: replace `for (int i = 0; i < N; i++)` with `for (int i = tid; i < N; i += stride)`.
- `encoder_forward` ports to a single `__global__` function with a 1-D index unflattened into `(b, t, c)`.

### What's next

**Chapter 11 — Memory Coalescing & Vectorized Loads.** GELU and encoder are bandwidth-bound. We'll see how the GPU's memory subsystem actually wants you to issue reads (32 threads in a warp accessing 32 *contiguous* floats = 1 transaction; non-contiguous = 32 transactions), and meet `Packed128` — the 128-bit vectorized-load type used everywhere in `llm.c`'s production kernels. We'll port `gelu_forward_kernel2` from `dev/cuda` and watch the bandwidth jump.

When you're ready, say **"proceed to Chapter 11"**.
